# UC1 unit tests

Runs the `pytest` suite from `Olist_DE_Practise_Jobs/tests/` against the pure
PySpark logic in `utilities/revenue_logic.py`. The suite validates the core
transformations - payment cleanup, 1-to-many rollup, ghost-order defence,
MoM / YoY / rolling-revenue windowing, and the Gold mart shape contracts -
against small synthetic DataFrames, with no DLT dependency.

Because `conftest.py` uses `SparkSession.builder.getOrCreate()`, the tests run
on the same cluster Spark that runs the DLT pipeline, so a green suite here
means the logic behind the pipeline is verified on the same runtime.

In [ ]:
import os
import sys

sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"

import pytest

try:
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    notebook_path = ctx.notebookPath().get()
    notebook_fs_path = (
        notebook_path
        if notebook_path.startswith("/Workspace/")
        else "/Workspace" + notebook_path
    )
    notebook_dir = os.path.dirname(notebook_fs_path)
except Exception:
    notebook_dir = os.getcwd()

package_dir = os.path.dirname(notebook_dir)
tests_dir = os.path.join(package_dir, "tests")
utilities_dir = os.path.join(package_dir, "utilities")

if utilities_dir not in sys.path:
    sys.path.insert(0, utilities_dir)

print(f"pytest version : {pytest.__version__}")
print(f"tests directory: {tests_dir}")
print(f"utilities path : {utilities_dir}")
assert os.path.isdir(tests_dir), f"tests directory not found at {tests_dir}"
assert os.path.isdir(utilities_dir), f"utilities directory not found at {utilities_dir}"

In [ ]:
exit_code = pytest.main([
    "-v",
    "--tb=short",
    "--no-header",
    "-p", "no:cacheprovider",
    "--import-mode=importlib",
    tests_dir,
])

assert exit_code == 0, f"pytest failed with exit code {exit_code}"